# APEX - Phase 1: Data Collection

**Goal:** understand the OpenF1 data before designing the dashboard around it.

We pick **one historical session and one driver**, pull every dataset we need,
inspect it, and save it locally so we never have to re-download.

**Test case:** 2024 Abu Dhabi Grand Prix - Race, **Max Verstappen (Red Bull, #1)**.

> **About OpenF1 access:** historical data is free and needs no account. But while a
> live F1 session is running, the whole API is locked until it ends (about 30 min
> after). If a cell below reports a *live session lock*, just run the notebook again
> once the session is over. Everything we fetch is cached to Parquet, so once it
> works you never pay the download cost again.

## 1. Import libraries

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can use config.py and src/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import pandas as pd
import numpy as np

import config
from src.data.api_client import OpenF1Client, LiveSessionLockError, is_api_open

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Target:", config.TARGET_DRIVER_NAME, "| year", config.TARGET_YEAR,
      "|", config.TARGET_COUNTRY, config.TARGET_SESSION_NAME)

Project root: f:\2_KIROKIRO\Projects\OpenF1
Target: Max Verstappen | year 2024 | United Arab Emirates Race


## 2. Connect to the OpenF1 API

We create one client. It handles rate limiting, caching, and the live-session lock.
First we probe whether the API is open right now.

In [2]:
client = OpenF1Client()

if is_api_open(client):
    print("OpenF1 is OPEN - historical data is reachable. We can fetch live.")
else:
    print("OpenF1 is LOCKED right now (a live session is likely in progress).")
    print("That's fine: cached data (if any) still loads. Re-run later to fetch.")

OpenF1 is OPEN - historical data is reachable. We can fetch live.


### A small helper

`fetch` wraps every call so a live lock does not crash the notebook. It returns the
cached copy if we already have it, otherwise tries the API, and on a lock it returns
`None` with a clear message.

In [3]:
def fetch(name, endpoint, params):
    """Fetch one dataset, cache it, and report its shape. Returns a DataFrame or None."""
    try:
        df = client.get(endpoint, params)
        print(f"[OK]   {name:<12} rows={len(df):>6}  cols={list(df.columns)[:8]}")
        return df
    except LiveSessionLockError as e:
        print(f"[LOCK] {name:<12} API locked by a live session - will fetch on a later run.")
        return None
    except Exception as e:
        print(f"[ERR]  {name:<12} {type(e).__name__}: {e}")
        return None

## 3. Retrieve meeting information

A *meeting* is a Grand Prix weekend. We find the 2024 Abu Dhabi meeting, then keep
its `meeting_key` to filter everything else.

In [4]:
meetings = fetch("meetings", "meetings",
                 {"year": config.TARGET_YEAR, "country_name": config.TARGET_COUNTRY})

meeting_key = None
if meetings is not None and not meetings.empty:
    display(meetings[["meeting_key", "meeting_name", "location", "date_start"]]
            if set(["meeting_name"]).issubset(meetings.columns) else meetings.head())
    meeting_key = int(meetings.iloc[0]["meeting_key"])
    print("meeting_key =", meeting_key)

[OK]   meetings     rows=     1  cols=['meeting_key', 'meeting_name', 'meeting_official_name', 'location', 'country_key', 'country_code', 'country_name', 'country_flag']


,meeting_key,meeting_name,location,date_start
0,1252,Abu Dhabi Grand Prix,Yas Island,2024-12-06T09:30:00+00:00


meeting_key = 1252


## 4. Retrieve session information

Each meeting has several *sessions* (Practice, Qualifying, Sprint, Race). We want the
**Race**, and we keep its `session_key` - the single most important id in the project.

In [5]:
sessions = fetch("sessions", "sessions",
                 {"year": config.TARGET_YEAR, "country_name": config.TARGET_COUNTRY,
                  "session_name": config.TARGET_SESSION_NAME})

session_key = None
if sessions is not None and not sessions.empty:
    cols = [c for c in ["session_key","session_name","date_start","date_end","circuit_short_name"]
            if c in sessions.columns]
    display(sessions[cols])
    session_key = int(sessions.iloc[0]["session_key"])
    print("session_key =", session_key)

[OK]   sessions     rows=     1  cols=['session_key', 'session_type', 'session_name', 'date_start', 'date_end', 'meeting_key', 'circuit_key', 'circuit_short_name']


,session_key,session_name,date_start,date_end,circuit_short_name
0,9662,Race,2024-12-08T13:00:00+00:00,2024-12-08T15:00:00+00:00,Yas Marina Circuit


session_key = 9662


## 5. Retrieve driver information

The `/drivers` endpoint lists everyone in the session, with names, numbers, team, and
team colour. We confirm Verstappen is there and grab his number (should be 1).

In [6]:
drivers = None
driver_number = config.TARGET_DRIVER_NUMBER
if session_key is not None:
    drivers = fetch("drivers", "drivers", {"session_key": session_key})
    if drivers is not None and not drivers.empty:
        cols = [c for c in ["driver_number","full_name","name_acronym","team_name","team_colour"]
                if c in drivers.columns]
        display(drivers[cols].sort_values("driver_number"))
        ver = drivers[drivers["full_name"].str.contains("Verstappen", case=False, na=False)]             if "full_name" in drivers.columns else drivers[drivers["driver_number"]==1]
        if not ver.empty:
            driver_number = int(ver.iloc[0]["driver_number"])
            print("Verstappen driver_number =", driver_number,
                  "| team_colour =", ver.iloc[0].get("team_colour"))

[OK]   drivers      rows=    20  cols=['meeting_key', 'session_key', 'driver_number', 'broadcast_name', 'full_name', 'name_acronym', 'team_name', 'team_colour']


,driver_number,full_name,name_acronym,team_name,team_colour
0,1,Max VERSTAPPEN,VER,Red Bull Racing,3671C6
1,4,Lando NORRIS,NOR,McLaren,FF8000
2,10,Pierre GASLY,GAS,Alpine,0093cc
3,11,Sergio PEREZ,PER,Red Bull Racing,3671C6
4,14,Fernando ALONSO,ALO,Aston Martin,229971
5,16,Charles LECLERC,LEC,Ferrari,E80020
6,18,Lance STROLL,STR,Aston Martin,229971
7,20,Kevin MAGNUSSEN,MAG,Haas F1 Team,B6BABD
8,22,Yuki TSUNODA,TSU,RB,6692FF
9,23,Alexander ALBON,ALB,Williams,64C4FF


Verstappen driver_number = 1 | team_colour = 3671C6


## 6. Retrieve lap data

`/laps` gives one row per lap for our driver: lap number, lap duration, sector times,
and speed-trap values. This is where we later find the fastest lap.

In [7]:
laps = None
if session_key is not None:
    laps = fetch("laps", "laps", {"session_key": session_key, "driver_number": driver_number})
    if laps is not None and not laps.empty:
        cols = [c for c in ["lap_number","lap_duration","duration_sector_1",
                            "duration_sector_2","duration_sector_3","st_speed","is_pit_out_lap"]
                if c in laps.columns]
        display(laps[cols].head(10))

[OK]   laps         rows=    58  cols=['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3']


,lap_number,lap_duration,duration_sector_1,duration_sector_2,duration_sector_3,st_speed,is_pit_out_lap
0,1,99.877,25.002,39.931,34.944,289,False
1,2,114.938,18.209,54.569,42.160,320,False
2,3,98.051,23.196,41.997,32.858,321,False
3,4,89.504,18.166,38.187,33.151,312,False
4,5,89.813,18.056,37.945,33.812,319,False
5,6,89.412,18.012,38.101,33.299,322,False
6,7,89.885,18.085,38.580,33.220,308,False
7,8,89.824,18.021,37.966,33.837,329,False
8,9,89.445,18.085,38.121,33.239,326,False
9,10,90.449,18.135,38.811,33.503,306,False


## 7. Retrieve car telemetry

`/car_data` is the heart of the dashboard: speed, throttle, brake, RPM, gear, DRS,
sampled at ~3.7 Hz. This is the **biggest** dataset, so it may take a few seconds.

In [8]:
car = None
if session_key is not None:
    car = fetch("car_data", "car_data", {"session_key": session_key, "driver_number": driver_number})
    if car is not None and not car.empty:
        cols = [c for c in ["date","speed","throttle","brake","rpm","n_gear","drs"]
                if c in car.columns]
        display(car[cols].head(10))
        print("samples:", len(car))

[OK]   car_data     rows= 33619  cols=['date', 'session_key', 'throttle', 'rpm', 'brake', 'speed', 'n_gear', 'driver_number']


,date,speed,throttle,brake,rpm,n_gear,drs
0,2024-12-08T12:06:49.419000+00:00,0,0,0,0,0,0
1,2024-12-08T12:06:49.739000+00:00,0,0,0,0,0,0
2,2024-12-08T12:06:49.899000+00:00,0,0,0,0,0,0
3,2024-12-08T12:06:50.299000+00:00,0,0,0,0,0,0
4,2024-12-08T12:06:50.499000+00:00,0,0,0,0,0,0
5,2024-12-08T12:06:50.659000+00:00,0,0,0,0,0,0
6,2024-12-08T12:06:50.819000+00:00,0,0,0,0,0,0
7,2024-12-08T12:06:51.099000+00:00,0,0,0,0,0,0
8,2024-12-08T12:06:51.299000+00:00,0,0,0,0,0,0
9,2024-12-08T12:06:51.499000+00:00,0,0,0,0,0,0


samples: 33619


## 8. Retrieve circuit location

`/location` gives the car's x/y/z position over time. Note: the coordinates use an
**arbitrary origin** - they are not aligned to a real map. We will normalise them later.

In [9]:
location = None
if session_key is not None:
    location = fetch("location", "location", {"session_key": session_key, "driver_number": driver_number})
    if location is not None and not location.empty:
        cols = [c for c in ["date","x","y","z"] if c in location.columns]
        display(location[cols].head(10))
        print("samples:", len(location))

[OK]   location     rows= 34421  cols=['date', 'session_key', 'x', 'y', 'meeting_key', 'driver_number', 'z']


,date,x,y,z
0,2024-12-08T12:06:49.443000+00:00,0,0,0
1,2024-12-08T12:06:49.683000+00:00,0,0,0
2,2024-12-08T12:06:49.843000+00:00,0,0,0
3,2024-12-08T12:06:50.123000+00:00,0,0,0
4,2024-12-08T12:06:50.283000+00:00,0,0,0
5,2024-12-08T12:06:51.323000+00:00,0,0,0
6,2024-12-08T12:06:51.483000+00:00,0,0,0
7,2024-12-08T12:06:51.683000+00:00,0,0,0
8,2024-12-08T12:06:51.904000+00:00,0,0,0
9,2024-12-08T12:06:52.144000+00:00,0,0,0


samples: 34421


## 9. Retrieve weather data

`/weather` is sampled less often than telemetry (roughly once a minute). We will later
match each replay moment to the *nearest* weather reading.

In [10]:
weather = None
if session_key is not None:
    weather = fetch("weather", "weather", {"session_key": session_key})
    if weather is not None and not weather.empty:
        display(weather.head(10))

[OK]   weather      rows=   148  cols=['date', 'session_key', 'rainfall', 'humidity', 'air_temperature', 'pressure', 'wind_direction', 'meeting_key']


,date,session_key,rainfall,humidity,air_temperature,pressure,wind_direction,meeting_key,track_temperature,wind_speed
0,2024-12-08T12:07:44.039000+00:00,9662,0,42.0,27.8,1017.3,292,1252,37.2,1.0
1,2024-12-08T12:08:44.037000+00:00,9662,0,43.0,27.8,1017.3,304,1252,37.1,2.7
2,2024-12-08T12:09:44.041000+00:00,9662,0,43.0,27.7,1017.2,324,1252,36.8,2.5
3,2024-12-08T12:10:44.057000+00:00,9662,0,43.0,27.7,1017.2,277,1252,36.8,2.0
4,2024-12-08T12:11:44.070000+00:00,9662,0,44.0,27.6,1017.2,308,1252,36.7,1.7
5,2024-12-08T12:12:44.074000+00:00,9662,0,44.0,27.6,1017.1,103,1252,36.2,2.4
6,2024-12-08T12:13:44.073000+00:00,9662,0,44.0,27.6,1017.3,315,1252,36.1,2.1
7,2024-12-08T12:14:44.074000+00:00,9662,0,44.0,27.6,1017.0,287,1252,36.1,2.4
8,2024-12-08T12:15:44.078000+00:00,9662,0,44.0,27.6,1017.1,310,1252,36.1,1.0
9,2024-12-08T12:16:44.089000+00:00,9662,0,43.0,27.7,1017.0,109,1252,36.1,2.5


## 10. Retrieve tire stints

`/stints` tells us which tire compound was on the car and for which laps. We use it to
estimate tire age at any lap.

In [11]:
stints = None
if session_key is not None:
    stints = fetch("stints", "stints", {"session_key": session_key, "driver_number": driver_number})
    if stints is not None and not stints.empty:
        display(stints)

[OK]   stints       rows=     2  cols=['meeting_key', 'session_key', 'stint_number', 'driver_number', 'lap_start', 'lap_end', 'compound', 'tyre_age_at_start']


,meeting_key,session_key,stint_number,driver_number,lap_start,lap_end,compound,tyre_age_at_start
0,1252,9662,1,1,1,29,MEDIUM,0
1,1252,9662,2,1,30,58,HARD,0


## 11. Initial data inspection

For each dataset we check: how many rows, which columns, the data types, and how many
missing values. This tells us what is usable before we design any panel around it.

In [12]:
datasets = {
    "meetings": meetings, "sessions": sessions, "drivers": drivers,
    "laps": laps, "car_data": car, "location": location,
    "weather": weather, "stints": stints,
}

for name, df in datasets.items():
    print("=" * 70)
    if df is None:
        print(f"{name}: not fetched yet (API locked or error).")
        continue
    print(f"{name}: shape={df.shape}")
    print("dtypes:")
    print(df.dtypes)
    na = df.isna().sum()
    na = na[na > 0]
    print("missing values:" , dict(na) if not na.empty else "none")

meetings: shape=(1, 18)
dtypes:
meeting_key              int64
meeting_name               str
meeting_official_name      str
location                   str
country_key              int64
country_code               str
country_name               str
country_flag               str
circuit_key              int64
circuit_short_name         str
circuit_type               str
circuit_info_url           str
circuit_image              str
gmt_offset                 str
date_start                 str
date_end                   str
year                     int64
is_cancelled              bool
dtype: object
missing values: none
sessions: shape=(1, 15)
dtypes:
session_key           int64
session_type            str
session_name            str
date_start              str
date_end                str
meeting_key           int64
circuit_key           int64
circuit_short_name      str
country_key           int64
country_code            str
country_name            str
location                str
gmt_off

## 12. Save raw datasets

We write each dataset to `data/raw/` as Parquet. Parquet keeps data types and is small
and fast. This is our **scrape-once** guarantee: once saved, later notebooks and the
dashboard load these files instead of hitting the API.

(The client already caches responses under `data/cache/`. Here we save a clean, named
copy under `data/raw/` that the rest of the project reads.)

In [13]:
saved = []
for name, df in datasets.items():
    if df is not None and not df.empty:
        path = config.RAW_DIR / f"{name}.parquet"
        df.to_parquet(path, index=False)
        saved.append((name, path.name, len(df)))

print("Saved to data/raw/:")
for name, fname, n in saved:
    print(f"  {name:<12} -> {fname:<20} ({n} rows)")

if not saved:
    print("Nothing saved yet - run again once the live session window closes.")

Saved to data/raw/:
  meetings     -> meetings.parquet     (1 rows)
  sessions     -> sessions.parquet     (1 rows)
  drivers      -> drivers.parquet      (20 rows)
  laps         -> laps.parquet         (58 rows)
  car_data     -> car_data.parquet     (33619 rows)
  location     -> location.parquet     (34421 rows)
  weather      -> weather.parquet      (148 rows)
  stints       -> stints.parquet       (2 rows)


## 13. Data limitations to remember

From this first look, note for later phases:

- **Location coordinates** use an arbitrary origin - normalise before drawing a track.
- **Weather** is low-frequency - always use the *nearest* reading, never claim exact.
- **Sampling rates differ** across endpoints - align by timestamp when joining.
- **No tire/brake temps, fuel, or engine temp** in OpenF1 - the reference screen shows
  these, but we must not fake them.
- A **live session** locks the API; rerun later if a fetch was blocked.

**Phase 1 done when:** every dataset above is fetched, inspected, and saved to
`data/raw/`, and these limitations are recorded.